# 🖼️ Qwen2.5-VL + SAM3 Image Segmentation Pipeline

**Supports:** single image file *or* a folder of images  

**Workflow:** this notebook assigns **stable open-vocab class ids** per Qwen prompt (shared **`open_vocab_state.json`** under `OUTPUT_ROOT`). Then use **`remap_open_vocab_to_yolo.ipynb`** (collapse to a small YOLO list) or **`standardize_open_vocab_extend_yolo.ipynb`** (keep / extend names). **`mine_qwen_taxonomy.ipynb`** helps audit prompts. **Step 10** validates polygons; train only after remap/extend.

**Outputs per image:**
- `combined_topN_sam3.png` — all prompts overlaid on the image
- `sam3_{prompt}.png` — one annotated image per prompt
- `labels/boxes/{image_stem}.txt` — YOLO **detection** labels (`class cx cy w h`, 5 cols by default)
- `labels/segments/{image_stem}.txt` — YOLO **segmentation** labels (`class x1 y1 x2 y2 …`, normalized polygons from SAM3 masks)
- `labels/boxes/classes.txt` (and under `segments/`) — **open-vocab** class names (line index = class id until remap/extend)
- `labels/boxes/yolo_dataset_template.yaml` — starter `data.yaml` for detection
- `labels/segments/yolo_seg_dataset_template.yaml` — starter `data.yaml` for **segment** train (`yolo segment train`)
- `labels/masks/{image_stem}_{prompt}_obj{id}.png` — binary mask PNGs (`obj{id}` = SAM3 **instance** id this frame, not YOLO class)
- `labels/metadata.json` — `objects_bbox` + `objects_segment` (`yolo_class_id`, `sam3_obj_id`, scores)
- `processing_log.json` — timing + `yolo_seg_validation` (per-image checks)
- `summary_report.json` — aggregated stats across all images

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"PyTorch built with CUDA version: {torch.version.cuda}")
    print(f"Current CUDA device name: {torch.cuda.get_device_name(0)}")
    import subprocess
    try:
        nvcc_version = subprocess.check_output(['nvcc', '--version']).decode('utf-8')
        print(f"System NVCC CUDA version:\n{nvcc_version}")
    except FileNotFoundError:
        print("nvcc not found. Cannot determine system CUDA version via nvcc.")
    except Exception as e:
        print(f"Error getting nvcc version: {e}")

## Step 1 — Install Dependencies

In [ ]:
# Run once per Colab session
!pip install -q transformers accelerate qwen-vl-utils opencv-python-headless kernels ipywidgets
# !pip install -q sam3  # replace with the correct install command if needed
print('✅ Dependencies installed')

In [ ]:
import transformers
transformers.__version__

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✅ Drive mounted')

## Step 2.1 — HuggingFace Login

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 3 — Configuration  *(edit this cell only)*

In [29]:
from pathlib import Path

# --- Notebook: state checkpoint + dry run + Step 10b (sync marker) ---
# ──────────────────────────────────────────────
# INPUT: flat folder / single file OR USS-PF batch (see USS_PF_DATASET)
# ──────────────────────────────────────────────
# USS-PF: INPUT_PATH = root with subfolders `106297_20190411_...`; only *Front* / *Rear*
# images are used; top view (VIDEO_USS_TOPVIEW, etc.) is skipped. Artifacts go to
# OUTPUT_ROOT / <leading_sequence_id> / <image_stem>/ ...
USS_PF_DATASET = True
# INPUT_PATH = "/Users/ral3ply/Google Drive/My Drive/auto-labeler/all/uss_pf"
# Quick local test (set USS_PF_DATASET = False):
INPUT_PATH = "/Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/inputs/uss_pf"

# Supported image extensions
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}

# Output root (flat mode: OUTPUT_ROOT/<stem>/… ; USS-PF: OUTPUT_ROOT/<seq_id>/<stem>/…)
OUTPUT_ROOT = Path("/Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/outputs/uss_pf")

# Pipeline settings
TOP_N = 4  # max dominant objects to detect

# Models
QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
SAM3_MODEL_ID = "facebook/sam3"

# Label format for bounding boxes
# 'yolo'  → normalised cx cy w h  (one .txt per image)
# 'coco'  → absolute x y w h      (one .txt per image)
BBOX_LABEL_FORMAT = "yolo"

# ── Open-vocab labels (no fixed YOLO taxonomy in this notebook) ───────────
# Class column = index in shared registry (OUTPUT_ROOT / OPEN_VOCAB_STATE_NAME).
# Map / standardize later: remap_open_vocab_to_yolo.ipynb or standardize_open_vocab_extend_yolo.ipynb
OPEN_VOCAB_STATE_NAME = "open_vocab_state.json"
OPEN_VOCAB_STATE_PATH: Path | None = None  # None = OUTPUT_ROOT / OPEN_VOCAB_STATE_NAME
YOLO_LABEL_APPEND_CONFIDENCE = False  # True = 6 bbox cols (debug only)

# YOLO segmentation export (polygons from SAM3 masks, Ultralytics segment label format)
YOLO_SEG_POLY_EPS_RATIO = 0.002  # Douglas–Peucker epsilon as fraction of contour perimeter
YOLO_SEG_MIN_MASK_AREA = 4.0  # skip tiny regions (pixels² in mask space)

# ── Scan limits (subset of a large dataset; checkpoint still resumes within this job list) ───
# USS-PF: only the first N immediate subfolders (sequence dirs), sorted by name; None = all.
MAX_USS_PF_SEQUENCE_FOLDERS: int | None = None  # e.g. 50 to pilot on 50 sequences under INPUT_PATH
# After collection: cap total image jobs (flat dir or USS); applied after folder cap; None = no cap.
MAX_IMAGE_JOBS: int | None = None  # e.g. 200 to stop after 200 images even if more were collected

# ── Pipeline checkpoint (JSON state machine: resume after pause / crash) ───
# One file tracks every image job: pending → running → done | error
PIPELINE_STATE_PATH = OUTPUT_ROOT / "pipeline_state.json"
RESET_PIPELINE_STATE = False   # True = re-queue all jobs pending (does not delete output folders)
RETRY_FAILED_JOBS = True       # True = re-queue jobs that previously ended in error
PIPELINE_DRY_RUN = False      # True = list pending jobs only (no Qwen/SAM3; state file not modified)

# ──────────────────────────────────────────────
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"INPUT_PATH  : {INPUT_PATH}")
print(f"OUTPUT_ROOT : {OUTPUT_ROOT}")
print(
    f"TOP_N={TOP_N}  BBOX_FORMAT={BBOX_LABEL_FORMAT}  labels=open-vocab"
    f"  vocab={OUTPUT_ROOT / OPEN_VOCAB_STATE_NAME}"
    f"  scan_caps: seq_folders={MAX_USS_PF_SEQUENCE_FOLDERS} max_jobs={MAX_IMAGE_JOBS}"
)

INPUT_PATH  : /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/inputs/uss_pf
OUTPUT_ROOT : /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/outputs/uss_pf
TOP_N=4  BBOX_FORMAT=yolo  YOLO nc=16


## Step 4 — Utility Functions

In [30]:
import json
import re
import time
import datetime
import numpy as np
import cv2
from pathlib import Path

# ── File collection ────────────────────────────────────────────────────────
def collect_images(input_path: str, extensions: set) -> list:
    """Return a list of image paths from a file or directory."""
    p = Path(input_path)
    if p.is_file():
        if p.suffix.lower() not in extensions:
            raise ValueError(f"File {p} is not a recognised image format: {extensions}")
        return [p]
    elif p.is_dir():
        images = sorted([f for f in p.rglob("*") if f.suffix.lower() in extensions])
        if not images:
            raise FileNotFoundError(f"No image files found in {p}")
        return images
    else:
        raise FileNotFoundError(f"INPUT_PATH not found: {p}")


def uss_pf_sequence_id(seq_folder: Path) -> str:
    """Leading digits before first underscore, e.g. 106297_20190411_... -> 106297."""
    m = re.match(r"^(\d+)_", seq_folder.name)
    if not m:
        raise ValueError(f"USS-PF folder name has no leading digits_: {seq_folder.name}")
    return m.group(1)


def collect_uss_pf_front_rear_jobs(
    dataset_root: Path,
    extensions: set,
    output_base: Path,
    *,
    max_sequence_folders: int | None = None,
) -> list[tuple[Path, Path]]:
    """
    Walk immediate subfolders of dataset_root; collect Front/Rear images only.
    Returns (image_path, output_root_for_that_image) with output_root = output_base / <sequence_id>.
    max_sequence_folders: keep only the first N sequence directories (sorted by name).
    """
    if not dataset_root.is_dir():
        raise FileNotFoundError(dataset_root)
    seq_dirs = [
        seq
        for seq in sorted(dataset_root.iterdir())
        if seq.is_dir() and not seq.name.startswith(".")
    ]
    if max_sequence_folders is not None:
        seq_dirs = seq_dirs[: int(max_sequence_folders)]
    jobs: list[tuple[Path, Path]] = []
    for seq in seq_dirs:
        seq_id = uss_pf_sequence_id(seq)
        out_root = (output_base / seq_id).resolve()
        for f in sorted(seq.iterdir()):
            if not f.is_file() or f.suffix.lower() not in extensions:
                continue
            nu = f.name.upper()
            if "TOPVIEW" in nu or nu.startswith("VIDEO_USS_TOP"):
                continue
            st = f.stem.upper()
            if "FRONT" not in st and "REAR" not in st:
                continue
            jobs.append((f.resolve(), out_root))
    if not jobs:
        raise FileNotFoundError(f"No Front/Rear images under {dataset_root}")
    return jobs

# ── Pipeline state machine (resume / checkpoint) ───────────────────────────
def pipeline_job_id(image_path: Path) -> str:
    """Stable key for JSON state (resolved absolute path, posix slashes)."""
    return image_path.resolve().as_posix()


def _pipeline_default_job(job_id: str, image_path: Path, output_root: Path) -> dict:
    return {
        "job_id": job_id,
        "image_path": str(image_path.resolve()),
        "output_root": str(output_root.resolve()),
        "status": "pending",
        "attempts": 0,
        "last_error": None,
        "started_at": None,
        "finished_at": None,
    }


def save_pipeline_state(state_path: Path, state: dict) -> None:
    state_path.parent.mkdir(parents=True, exist_ok=True)
    state["updated_at"] = datetime.datetime.now().isoformat()
    tmp = state_path.with_suffix(".tmp.json")
    tmp.write_text(json.dumps(state, indent=2, ensure_ascii=False))
    tmp.replace(state_path)


def load_merge_pipeline_state(
    state_path: Path,
    image_paths: list,
    output_root: Path,
    *,
    reset: bool,
    retry_failed: bool,
    persist: bool = True,
    job_output_roots: dict | None = None,
) -> dict:
    """
    Load existing JSON or start fresh. Aligns `order` with the current `image_paths`
    scan; merges statuses for known job_ids. Any prior `running` job is reset to
    `pending` (kernel died mid-run). If `retry_failed`, `error` becomes `pending`.
    If persist is False, returns merged state but does not write state_path (dry run).
    job_output_roots: optional map job_id -> Path for per-image OUTPUT_ROOT (USS-PF layout).
    """
    order = [pipeline_job_id(p) for p in image_paths]
    old_jobs: dict = {}
    if state_path.exists() and not reset:
        try:
            old_jobs = json.loads(state_path.read_text()).get("jobs", {})
        except Exception:
            old_jobs = {}

    jobs: dict = {}
    for p in image_paths:
        jid = pipeline_job_id(p)
        out_r = output_root
        if job_output_roots and jid in job_output_roots:
            out_r = Path(job_output_roots[jid])
        if reset or jid not in old_jobs:
            entry = _pipeline_default_job(jid, p, out_r)
        else:
            entry = dict(old_jobs[jid])
            entry["job_id"] = jid
            entry["image_path"] = str(p.resolve())
            if entry.get("status") == "running":
                entry["status"] = "pending"
            if entry.get("status") == "error" and retry_failed:
                entry["status"] = "pending"
                entry["last_error"] = None
            if entry.get("status") != "done":
                entry["output_root"] = str(out_r.resolve())
        jobs[jid] = entry

    state = {"schema_version": 1, "order": order, "jobs": jobs}
    if persist:
        save_pipeline_state(state_path, state)
    return state


def pending_pipeline_job_ids(state: dict) -> list:
    return [
        jid
        for jid in state["order"]
        if state["jobs"][jid]["status"] in ("pending", "running")
    ]


def update_job_after_run(state: dict, job_id: str, log: dict, state_path: Path) -> None:
    j = state["jobs"][job_id]
    if log.get("status") == "done":
        j["status"] = "done"
        j["last_error"] = None
    else:
        j["status"] = "error"
        j["last_error"] = (log.get("error") or str(log))[:8000]
    j["finished_at"] = datetime.datetime.now().isoformat()
    save_pipeline_state(state_path, state)

# ── Output directory layout ────────────────────────────────────────────────
def make_image_output_dir(output_root: Path, image_path: Path) -> dict:
    """Create and return all sub-directory paths for one image."""
    slug = image_path.stem.replace(" ", "_")
    base = output_root / slug
    dirs = {
        "base":        base,
        "annotated":   base / "annotated",
        "label_boxes": base / "labels" / "boxes",
        "label_segments": base / "labels" / "segments",
        "label_masks": base / "labels" / "masks",
        "label_meta":  base / "labels",
    }
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    return dirs

# ── Image reader ───────────────────────────────────────────────────────────
def read_image(path: str) -> tuple:
    """
    Load an image and return it as a list containing one RGB numpy frame
    (matches the interface expected by SAM3 video session).
    """
    img_bgr = cv2.imread(str(path))
    if img_bgr is None:
        raise RuntimeError(f"Could not open image: {path}")
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    return [img_rgb], {"height": h, "width": w}

# ── JSON helpers ──────────────────────────────────────────────────────────
def extract_json_block(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```json\s*(\{.*?\})\s*```", text, flags=re.S)
    if m:
        try: return json.loads(m.group(1))
        except Exception: pass
    m = re.search(r"(\{.*\})", text, flags=re.S)
    if m:
        try: return json.loads(m.group(1))
        except Exception: pass
    raise ValueError(f"Could not parse JSON from:\n{text[:300]}")

# ── Timing context manager ────────────────────────────────────────────────
class Timer:
    def __init__(self, label=""):
        self.label   = label
        self.elapsed = 0.0
    def __enter__(self):
        self._start = time.perf_counter()
        return self
    def __exit__(self, *_):
        self.elapsed = time.perf_counter() - self._start
        print(f"  ⏱  {self.label}: {self.elapsed:.2f}s")

# ── Colour palette ────────────────────────────────────────────────────────
def color_from_text(text: str) -> tuple:
    rng = np.random.default_rng(abs(hash(text)) % (2**32))
    return tuple(int(x) for x in rng.integers(60, 230, size=3))

print('✅ Utilities loaded')

✅ Utilities loaded


## Step 5 — Load Models  *(run once)*

In [31]:
import gc
import torch
import transformers
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    Sam3VideoModel,
    Sam3VideoProcessor,
)

device     = "cuda" if torch.cuda.is_available() else "cpu"
qwen_dtype = torch.bfloat16 if device == "cuda" else torch.float32
sam3_dtype = torch.bfloat16 if device == "cuda" else torch.float32

print("transformers version:", transformers.__version__)
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Qwen
print("\nLoading Qwen...")
with Timer("Qwen load"):
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        QWEN_MODEL_ID,
        torch_dtype=qwen_dtype,
        device_map="auto",
    )
    qwen_processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)

# SAM3 Video — reused here for single-frame (image) inference
# We pass each image as a 1-frame video to the session API
print("\nLoading SAM3...")
with Timer("SAM3 load"):
    sam3_model = Sam3VideoModel.from_pretrained(
        SAM3_MODEL_ID
    ).to(device=device, dtype=sam3_dtype)
    sam3_processor = Sam3VideoProcessor.from_pretrained(SAM3_MODEL_ID)

# Workaround: run_detection does get_text_features(..., return_dict=True).pooler_output.
# If the vision-language backbone returns a plain Tensor, wrap it. Patch the detector class
# (idempotent) so every code path sees .pooler_output.
from types import SimpleNamespace

_GET_TEXT_UNPATCH = "_orig_get_text_features_sam3"


def _normalize_sam3_text_out(out):
    if torch.is_tensor(out) or isinstance(out, torch.Tensor):
        return SimpleNamespace(pooler_output=out)
    if hasattr(out, "pooler_output") and getattr(out, "pooler_output", None) is not None:
        return out
    if hasattr(out, "text_embeds") and getattr(out, "text_embeds", None) is not None:
        return SimpleNamespace(pooler_output=out.text_embeds)
    raise TypeError(f"Unexpected get_text_features return type: {type(out)}")


if hasattr(sam3_model, "detector_model") and hasattr(sam3_model.detector_model, "get_text_features"):
    if "get_text_features" in sam3_model.detector_model.__dict__:
        delattr(sam3_model.detector_model, "get_text_features")
    _dm_cls = type(sam3_model.detector_model)
    if not hasattr(_dm_cls, _GET_TEXT_UNPATCH):
        setattr(_dm_cls, _GET_TEXT_UNPATCH, _dm_cls.get_text_features)
    _unpatched = getattr(_dm_cls, _GET_TEXT_UNPATCH)

    def _get_text_features_fixed(self, *args, **kwargs):
        return _normalize_sam3_text_out(_unpatched(self, *args, **kwargs))

    _dm_cls.get_text_features = _get_text_features_fixed
    print("✅ Applied SAM3 text-feature compatibility patch (class-level, idempotent)")


print("\n✅ Models ready")

transformers version: 5.3.0.dev0
Device : cpu

Loading Qwen...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


  ⏱  Qwen load: 24.21s

Loading SAM3...


Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

  ⏱  SAM3 load: 3.07s
✅ Applied SAM3 text-feature compatibility patch

✅ Models ready


## Step 6 — Inference Functions

In [32]:
from qwen_vl_utils import process_vision_info

# ── Qwen: extract dominant object prompts from an image ───────────────────
def run_qwen(image_path: str, top_n: int) -> tuple:
    """
    Ask Qwen to identify the top N dominant objects in a single image.
    Uses {"type": "image"} instead of {"type": "video"}.
    """
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": f"file://{image_path}"},
            {"type": "text", "text": (
                f"Identify the top {top_n} dominant physical objects seen in this image. "
                "Prefer concrete visible objects, not events or abstract concepts. "
                "Use short noun phrases suitable as segmentation prompts. "
                "Return STRICT JSON only with this schema:\n"
                "{\n"
                '  "dominant_objects": [\n'
                '    {"name": "short noun phrase", "reason": "short reason", "frequency": "high|medium|low"}\n'
                "  ]\n"
                "}\n"
                f"Return at most {top_n} objects."
            )},
        ],
    }]

    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt"
    ).to(qwen_model.device)

    gen_ids = qwen_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen_ids)]
    raw_text = qwen_processor.batch_decode(trimmed, skip_special_tokens=True,
                                           clean_up_tokenization_spaces=False)[0]

    qwen_json = extract_json_block(raw_text)
    prompts   = []
    for item in qwen_json.get("dominant_objects", [])[:top_n]:
        name = str(item.get("name", "")).strip().lower()
        if name and name not in prompts:
            prompts.append(name)

    return qwen_json, prompts

# ── SAM3: segment a single image with given prompts ───────────────────────
def run_sam3_image(image_frame: np.ndarray, prompts: list) -> dict:
    """
    Wraps the image as a 1-frame video and runs SAM3.
    Returns the processed result dict for frame 0.
    """
    assert len(prompts) > 0, "No prompts to segment"

    # Pass the single frame as a list — SAM3 video session handles 1-frame input
    sam_session = sam3_processor.init_video_session(
        video=[image_frame],
        inference_device=device,
        processing_device="cpu",
        video_storage_device="cpu",
        dtype=sam3_dtype,
    )

    sam_session = sam3_processor.add_text_prompt(
        inference_session=sam_session,
        text=prompts,
    )

    # Collect the single frame result
    result = None
    for model_outputs in sam3_model.propagate_in_video_iterator(
        inference_session=sam_session
    ):
        result = sam3_processor.postprocess_outputs(sam_session, model_outputs)

    return result

print('✅ Inference functions ready')

✅ Inference functions ready


## Step 7 — Label & Visualisation Export Functions

In [33]:
import json  # open-vocab state + process_image metadata

# ── Tensor → numpy helper ─────────────────────────────────────────────────
def to_np(x):
    return x.cpu().numpy() if hasattr(x, "cpu") else np.asarray(x)

# ── Visualisation helpers ─────────────────────────────────────────────────
def overlay_mask(image_bgr, mask_bool, color, alpha=0.40):
    out = image_bgr.copy()
    out[mask_bool] = (
        out[mask_bool] * (1 - alpha) + np.array(color, dtype=np.float32) * alpha
    ).astype(np.uint8)
    return out

def draw_box_label(img, box_xyxy, label, color):
    x1, y1, x2, y2 = map(int, box_xyxy)
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    cv2.putText(img, label, (x1, max(20, y1-8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)

def draw_mask_contour(img, mask_bool, color, thickness=2):
    # Convert boolean mask to uint8 for findContours
    mask_uint8 = (mask_bool * 255).astype(np.uint8)
    contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(img, contours, -1, color, thickness)

# ── Open-vocab: stable ids per unseen prompt key ───────────────────────────
def _open_vocab_norm_key(prompt: str) -> str:
    """Stable key: lowercase, strip, collapse spaces — same phrase = same id (e.g. Yellow Cab == yellow cab)."""
    return " ".join(prompt.strip().lower().split())


def load_open_vocab_state(state_path: Path) -> dict:
    if state_path.is_file():
        try:
            data = json.loads(state_path.read_text())
            if isinstance(data, dict) and "id_to_prompt" in data and "key_to_id" in data:
                return data
        except Exception:
            pass
    return {"schema_version": 1, "id_to_prompt": [], "key_to_id": {}}


def save_open_vocab_state(state_path: Path, st: dict) -> None:
    state_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = state_path.with_suffix(".tmp.json")
    tmp.write_text(json.dumps(st, indent=2, ensure_ascii=False))
    tmp.replace(state_path)


def merge_open_vocab_for_prompts(st: dict, prompts: list[str]) -> dict[str, int]:
    """Extend registry with any new prompts; return key_to_id map (mutates st)."""
    id_to = list(st["id_to_prompt"])
    k2i = {str(k): int(v) for k, v in st.get("key_to_id", {}).items()}
    for p in prompts:
        k = _open_vocab_norm_key(p)
        if not k:
            continue
        if k not in k2i:
            k2i[k] = len(id_to)
            id_to.append(k)  # same canonical string as lookup key (all lower, no dup casing)
    st["id_to_prompt"] = id_to
    st["key_to_id"] = k2i
    return k2i


def open_vocab_state_path() -> Path:
    """Single JSON registry for the whole dataset (under OUTPUT_ROOT unless overridden)."""
    if OPEN_VOCAB_STATE_PATH is not None:
        return Path(OPEN_VOCAB_STATE_PATH)
    return OUTPUT_ROOT / OPEN_VOCAB_STATE_NAME

# ── Bounding-box label writer ─────────────────────────────────────────────
def save_bbox_labels(
    image_stem: str,
    result: dict,
    prompts: list,
    label_dir: Path,
    img_hw: tuple,
    fmt: str = "yolo",
    *,
    open_vocab_key_to_id: dict[str, int],
    export_class_names: list[str],
):
    """
    Writes one .txt label file per image (open-vocab class id = index in export_class_names).
    SAM3 obj_id is stored in metadata only (per-image instance key).
    Default: 5 columns (class + cx cy w h). Optional 6th = confidence for debug.
    """
    H, W  = img_hw
    masks   = to_np(result["masks"])
    boxes   = to_np(result["boxes"])
    scores  = to_np(result["scores"])
    obj_ids = to_np(result["object_ids"])
    p2obj   = result["prompt_to_obj_ids"]

    obj_idx = {int(oid): i for i, oid in enumerate(obj_ids)}
    lines   = []
    meta    = []

    for prompt in prompts:
        pk = _open_vocab_norm_key(prompt)
        if not pk or pk not in open_vocab_key_to_id:
            continue
        yolo_cid = open_vocab_key_to_id[pk]

        raw_ids = to_np(p2obj.get(prompt, []))
        for obj_id in [int(x) for x in raw_ids]:
            if obj_id not in obj_idx:
                continue
            i   = obj_idx[obj_id]
            x1, y1, x2, y2 = boxes[i]
            score = float(scores[i])

            if fmt == "yolo":
                cx = ((x1 + x2) / 2) / W
                cy = ((y1 + y2) / 2) / H
                bw = (x2 - x1) / W
                bh = (y2 - y1) / H
                row = f"{yolo_cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
                if YOLO_LABEL_APPEND_CONFIDENCE:
                    row += f" {score:.4f}"
                lines.append(row)
            else:  # coco absolute
                row = f"{yolo_cid} {x1:.1f} {y1:.1f} {x2-x1:.1f} {y2-y1:.1f}"
                if YOLO_LABEL_APPEND_CONFIDENCE:
                    row += f" {score:.4f}"
                lines.append(row)

            meta.append({
                "yolo_class_id": yolo_cid,
                "yolo_class_name": (
                    export_class_names[yolo_cid]
                    if 0 <= yolo_cid < len(export_class_names)
                    else None
                ),
                "prompt": prompt,
                "sam3_obj_id": obj_id,
                "score": score,
                "box_xyxy": [float(x1), float(y1), float(x2), float(y2)],
            })

    txt_path = label_dir / f"{image_stem}.txt"
    txt_path.write_text("\n".join(lines))
    return meta

# ── Binary mask writer ────────────────────────────────────────────────────
def save_mask_pngs(image_stem: str, result: dict, prompts: list, mask_dir: Path):
    """Saves one binary PNG per (prompt, obj_id) for the image."""
    masks   = to_np(result["masks"])
    obj_ids = to_np(result["object_ids"])
    p2obj   = result["prompt_to_obj_ids"]
    obj_idx = {int(oid): i for i, oid in enumerate(obj_ids)}

    paths = []
    for prompt in prompts:
        raw_ids = to_np(p2obj.get(prompt, []))
        slug    = prompt.replace(" ", "_")
        for obj_id in [int(x) for x in raw_ids]:
            if obj_id not in obj_idx:
                continue
            i     = obj_idx[obj_id]
            m     = (masks[i].astype(np.uint8)) * 255
            fpath = mask_dir / f"{image_stem}_{slug}_obj{obj_id}.png"
            cv2.imwrite(str(fpath), m)
            paths.append(str(fpath))
    return paths

# ── YOLO segmentation labels (Ultralytics: class + normalized polygon) ───
def mask_to_normalized_polygon(mask_hw: np.ndarray, W: int, H: int) -> list[float] | None:
    """Largest outer contour → flat [x1,y1,x2,y2,…] in 0–1; or None."""
    m = np.asarray(mask_hw)
    if m.dtype != np.uint8:
        m = (m > 0.5).astype(np.uint8)
    else:
        m = (m > 127).astype(np.uint8)
    m = (m * 255).astype(np.uint8)
    if float(m.sum()) < YOLO_SEG_MIN_MASK_AREA:
        return None
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    cnt = max(contours, key=cv2.contourArea)
    if cv2.contourArea(cnt) < YOLO_SEG_MIN_MASK_AREA:
        return None
    peri = cv2.arcLength(cnt, True)
    eps = max(YOLO_SEG_POLY_EPS_RATIO * peri, 1e-6)
    cnt = cv2.approxPolyDP(cnt, eps, True)
    if len(cnt) < 3:
        return None
    pts = cnt.reshape(-1, 2).astype(np.float64)
    pts[:, 0] = np.clip(pts[:, 0] / float(W), 0.0, 1.0)
    pts[:, 1] = np.clip(pts[:, 1] / float(H), 0.0, 1.0)
    flat = pts.flatten()
    if flat.size < 6:
        return None
    return flat.tolist()


def save_yolo_segment_labels(
    image_stem: str,
    result: dict,
    prompts: list,
    segment_dir: Path,
    img_hw: tuple,
    *,
    open_vocab_key_to_id: dict[str, int],
):
    """
    One .txt per image: each line `class x1 y1 x2 y2 ...` (normalized polygon).
    Open-vocab: class id = index in shared registry (export_class_names in metadata).
    """
    H, W = img_hw
    masks = to_np(result["masks"])
    boxes = to_np(result["boxes"])
    scores = to_np(result["scores"])
    obj_ids = to_np(result["object_ids"])
    p2obj = result["prompt_to_obj_ids"]
    obj_idx = {int(oid): i for i, oid in enumerate(obj_ids)}
    lines = []
    meta_seg = []

    for prompt in prompts:
        pk = _open_vocab_norm_key(prompt)
        if not pk or pk not in open_vocab_key_to_id:
            continue
        yolo_cid = open_vocab_key_to_id[pk]

        raw_ids = to_np(p2obj.get(prompt, []))
        for obj_id in [int(x) for x in raw_ids]:
            if obj_id not in obj_idx:
                continue
            i = obj_idx[obj_id]
            poly = mask_to_normalized_polygon(masks[i], W, H)
            if poly is None:
                continue
            row = f"{yolo_cid} " + " ".join(f"{p:.6f}" for p in poly)
            lines.append(row)
            x1, y1, x2, y2 = boxes[i]
            meta_seg.append(
                {
                    "yolo_class_id": yolo_cid,
                    "prompt": prompt,
                    "sam3_obj_id": obj_id,
                    "score": float(scores[i]),
                    "polygon_points": len(poly) // 2,
                    "box_xyxy": [float(x1), float(y1), float(x2), float(y2)],
                }
            )

    out = segment_dir / f"{image_stem}.txt"
    out.write_text("\n".join(lines))
    return meta_seg


# ── YOLO-seg artifact validation (Ultralytics segment label format) ─────
def validate_yolo_seg_label_line(line: str, nc: int) -> tuple[bool, str]:
    parts = line.split()
    if len(parts) < 7:
        return False, f"need class + ≥3 (x,y) pairs (got {len(parts)} tokens)"
    try:
        cid = int(float(parts[0]))
    except ValueError:
        return False, "class id must be numeric"
    if cid < 0 or cid >= nc:
        return False, f"class {cid} not in [0, {nc - 1}]"
    try:
        coords = [float(x) for x in parts[1:]]
    except ValueError:
        return False, "non-numeric coordinates"
    if len(coords) % 2 != 0:
        return False, "coordinate count must be even"
    if len(coords) < 6:
        return False, "need ≥ 3 polygon vertices"
    for i, v in enumerate(coords):
        if v < -1e-6 or v > 1.0 + 1e-6:
            return False, f"coord[{i}]={v:g} outside [0,1] (YOLO norm space)"
    return True, "ok"


def validate_yolo_seg_image_folder(
    base: Path, image_stem: str, nc: int, class_names: list[str]
) -> list[str]:
    """Return list of issue strings; empty => OK for segment labels + classes.txt."""
    issues: list[str] = []
    seg_dir = base / "labels" / "segments"
    seg_txt = seg_dir / f"{image_stem}.txt"
    cls_p = seg_dir / "classes.txt"

    if not seg_txt.is_file():
        issues.append(f"missing segment label: {seg_txt}")
        return issues

    if cls_p.is_file():
        nlines = len([x for x in cls_p.read_text().splitlines() if x.strip()])
        if nlines != len(class_names):
            issues.append(f"segments/classes.txt: {nlines} lines, expected {len(class_names)}")
        if nlines != nc:
            issues.append(f"segments/classes.txt lines ({nlines}) != nc ({nc})")

    raw = seg_txt.read_text().strip()
    if not raw:
        return issues

    for li, line in enumerate(raw.splitlines(), 1):
        line = line.strip()
        if not line:
            continue
        ok, msg = validate_yolo_seg_label_line(line, nc)
        if not ok:
            issues.append(f"{seg_txt.name} line {li}: {msg}")
    return issues


def _read_seg_nc_from_classes(seg_dir: Path) -> int:
    cls_p = seg_dir / "classes.txt"
    if not cls_p.is_file():
        return 0
    return len([ln for ln in cls_p.read_text().splitlines() if ln.strip()])


def _iter_yolo_seg_run_dirs(output_root: Path):
    """Flat OUTPUT_ROOT/<stem>/ or nested OUTPUT_ROOT/<seq>/<stem>/."""
    for sub in sorted(output_root.iterdir()):
        if not sub.is_dir():
            continue
        if (sub / "labels" / "segments").is_dir():
            yield sub.name, sub
            continue
        for nest in sorted(sub.iterdir()):
            if not nest.is_dir():
                continue
            if (nest / "labels" / "segments").is_dir():
                yield f"{sub.name}/{nest.name}", nest


def validate_yolo_seg_output_root(output_root: Path, nc: int | None = None) -> dict:
    """Scan run folders; if nc is None, use each folder's classes.txt line count."""
    summary = {"folders": 0, "ok": 0, "with_issues": 0, "details": []}
    for label, run_dir in _iter_yolo_seg_run_dirs(output_root):
        seg_dir = run_dir / "labels" / "segments"
        if not seg_dir.is_dir():
            summary["details"].append({"folder": label, "issues": ["no labels/segments/"]})
            summary["with_issues"] += 1
            continue
        summary["folders"] += 1
        nc_run = _read_seg_nc_from_classes(seg_dir) if nc is None else nc
        if nc_run <= 0:
            summary["with_issues"] += 1
            summary["details"].append({"folder": label, "issues": ["no classes.txt or empty — cannot validate class ids"]})
            continue
        label_txts = [p for p in seg_dir.glob("*.txt") if p.name != "classes.txt"]
        folder_issues: list[str] = []
        if not label_txts:
            folder_issues.append("no *.txt label file (only classes.txt?)")
        for p in label_txts:
            raw = p.read_text().strip()
            if not raw:
                continue
            for li, line in enumerate(raw.splitlines(), 1):
                line = line.strip()
                if not line:
                    continue
                ok, msg = validate_yolo_seg_label_line(line, nc_run)
                if not ok:
                    folder_issues.append(f"{p.name} L{li}: {msg}")
        if folder_issues:
            summary["with_issues"] += 1
            summary["details"].append({"folder": label, "issues": folder_issues})
        else:
            summary["ok"] += 1
    return summary


# ── Combined annotated image (all prompts) ────────────────────────────────
def render_combined_image(image_rgb, result, prompts, out_path):
    vis    = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    colors = {p: color_from_text(p) for p in prompts}

    masks   = to_np(result["masks"])
    boxes   = to_np(result["boxes"])
    scores  = to_np(result["scores"])
    obj_ids = to_np(result["object_ids"])
    p2obj   = result["prompt_to_obj_ids"]
    oidx    = {int(oid): i for i, oid in enumerate(obj_ids)}

    for prompt in prompts:
        color = colors[prompt]
        for obj_id in [int(x) for x in to_np(p2obj.get(prompt, []))]:
            if obj_id not in oidx: continue
            i = oidx[obj_id]
            vis = overlay_mask(vis, masks[i].astype(bool), color, 0.35)
            draw_box_label(vis, boxes[i],
                           f"{prompt}|id={obj_id}|{float(scores[i]):.2f}", color)
            draw_mask_contour(vis, masks[i].astype(bool), color, thickness=2)

    for y_off, prompt in enumerate(prompts, start=1):   # legend
        cv2.putText(vis, prompt, (10, 28*y_off),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, colors[prompt], 2, cv2.LINE_AA)

    cv2.imwrite(str(out_path), vis)

# ── Per-prompt annotated images ────────────────────────────────────────────
def render_per_prompt_images(image_rgb, result, prompts, annotated_dir):
    colors  = {p: color_from_text(p) for p in prompts}
    base    = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    masks   = to_np(result["masks"])
    boxes   = to_np(result["boxes"])
    scores  = to_np(result["scores"])
    obj_ids = to_np(result["object_ids"])
    p2obj   = result["prompt_to_obj_ids"]
    oidx    = {int(oid): i for i, oid in enumerate(obj_ids)}

    for prompt in prompts:
        vis   = base.copy()
        color = colors[prompt]
        for obj_id in [int(x) for x in to_np(p2obj.get(prompt, []))]:
            if obj_id not in oidx: continue
            i = oidx[obj_id]
            vis = overlay_mask(vis, masks[i].astype(bool), color, 0.40)
            draw_box_label(vis, boxes[i],
                           f"{prompt}|id={obj_id}|{float(scores[i]):.2f}", color)
            draw_mask_contour(vis, masks[i].astype(bool), color, thickness=2)

        cv2.putText(vis, f"Prompt: {prompt}", (10, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)
        out_path = annotated_dir / f"sam3_{prompt.replace(' ', '_')}.png"
        cv2.imwrite(str(out_path), vis)

print('✅ Export functions ready')

✅ Export functions ready


## Step 8 — Per-Image Pipeline

In [34]:
def process_image(image_path: Path, output_root: Path,
                  top_n: int, bbox_fmt: str) -> dict:
    """
    Full pipeline for one image.
    Returns a log dict with timing, paths, and stats.
    """
    log = {
        "image_path":  str(image_path),
        "image_stem":  image_path.stem,
        "started_at":  datetime.datetime.now().isoformat(),
        "status":      "started",
        "error":       None,
        "prompts":     [],
        "t_read_s":    0.0,
        "t_qwen_s":    0.0,
        "t_sam3_s":    0.0,
        "t_export_s":  0.0,
        "t_total_s":   0.0,
        "output_dir":  "",
        "yolo_seg_validation": [],
    }
    wall_start = time.perf_counter()

    try:
        dirs = make_image_output_dir(output_root, image_path)
        log["output_dir"] = str(dirs["base"])

        # ── 1. Read image ──────────────────────────────────────────────────
        print(f"\n{'='*60}")
        print(f"IMAGE : {image_path.name}")
        print(f"{'='*60}")
        with Timer("Read image") as t:
            frames, meta = read_image(str(image_path))
        log["t_read_s"] = t.elapsed
        image_frame    = frames[0]   # single RGB numpy array
        H, W           = meta["height"], meta["width"]
        print(f"  Size: {W}×{H}")

        # ── 2. Qwen prompt extraction ──────────────────────────────────────
        with Timer("Qwen inference") as t:
            qwen_json, prompts = run_qwen(str(image_path), top_n)
        log["t_qwen_s"] = t.elapsed
        log["prompts"]  = prompts
        print(f"  Prompts: {prompts}")

        # Save Qwen output
        (dirs["base"] / "qwen_output.json").write_text(
            json.dumps({"qwen_raw": qwen_json, "prompts": prompts}, indent=2))

        # Open-vocab class list (shared registry under OUTPUT_ROOT)
        _ov_path = open_vocab_state_path()
        _st = load_open_vocab_state(_ov_path)
        ov_key_to_id = merge_open_vocab_for_prompts(_st, prompts)
        save_open_vocab_state(_ov_path, _st)
        export_names = list(_st["id_to_prompt"])
        _cls_txt = "\n".join(export_names)
        (dirs["label_boxes"] / "classes.txt").write_text(_cls_txt)
        (dirs["label_segments"] / "classes.txt").write_text(_cls_txt)

        # ── 3. SAM3 segmentation ───────────────────────────────────────────
        with Timer("SAM3 inference") as t:
            result = run_sam3_image(image_frame, prompts)
        log["t_sam3_s"] = t.elapsed

        # ── 4. Free Qwen GPU memory before export ──────────────────────────
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        # ── 5. Export labels + annotated images ────────────────────────────
        with Timer("Export") as t:
            stem = image_path.stem

            # Bounding-box labels
            all_meta = save_bbox_labels(
                stem,
                result,
                prompts,
                dirs["label_boxes"],
                (H, W),
                bbox_fmt,
                open_vocab_key_to_id=ov_key_to_id,
                export_class_names=export_names,
            )

            # YOLO segmentation polygons (Ultralytics segment train)
            seg_meta = save_yolo_segment_labels(
                stem,
                result,
                prompts,
                dirs["label_segments"],
                (H, W),
                open_vocab_key_to_id=ov_key_to_id,
            )

            # Binary mask PNGs
            save_mask_pngs(stem, result, prompts, dirs["label_masks"])

            # Metadata JSON
            _nc_export = len(export_names)
            yolo_meta = {
                "image": str(image_path),
                "qwen_prompts_this_image": prompts,
                "bbox_format": bbox_fmt,
                "label_export_mode": "open_vocab",
                "yolo_nc": _nc_export,
                "yolo_class_names": list(export_names),
                "open_vocab_state_path": str(_ov_path),
                "label_append_confidence": YOLO_LABEL_APPEND_CONFIDENCE,
                "id_note": (
                    "sam3_obj_id = per-image instance; class column = open-vocab id; "
                    "remap/extend in remap_open_vocab_to_yolo.ipynb or standardize_open_vocab_extend_yolo.ipynb"
                ),
                "objects_bbox": all_meta,
                "objects_segment": seg_meta,
                "segment_label_file": str(dirs["label_segments"] / f"{stem}.txt"),
            }
            (dirs["label_meta"] / "metadata.json").write_text(json.dumps(yolo_meta, indent=2))
            tmpl = (
                "# Template for Ultralytics — set `path` and image folders\n"
                "# NOTE: open-vocab class ids — remap/extend before yolo train.\n"
                + f"path: .\n"
                + f"train: images/train\n"
                + f"val: images/val\n"
                + f"nc: {_nc_export}\n"
                + "names:\n"
            )
            for i, n in enumerate(export_names):
                tmpl += f"  {i}: {n}\n"
            (dirs["label_boxes"] / "yolo_dataset_template.yaml").write_text(tmpl)
            seg_tmpl = (
                "# Ultralytics YOLO segmentation — use: yolo segment train data=...\n"
                "# Point train/val to image folders; labels = same stem .txt with polygon rows.\n"
                "# Each label line: class x1 y1 x2 y2 ... (normalized 0-1).\n"
                "# NOTE: open-vocab class ids — remap/extend before training.\n"
                + f"path: .\n"
                + f"train: images/train\n"
                + f"val: images/val\n"
                + f"nc: {_nc_export}\n"
                + "names:\n"
            )
            for i, n in enumerate(export_names):
                seg_tmpl += f"  {i}: {n}\n"
            (dirs["label_segments"] / "yolo_seg_dataset_template.yaml").write_text(seg_tmpl)

            v_issues = validate_yolo_seg_image_folder(
                dirs["base"], stem, _nc_export, export_names
            )
            log["yolo_seg_validation"] = v_issues
            if v_issues:
                for msg in v_issues[:8]:
                    print(f"  ⚠️ YOLO-seg check: {msg}")
                if len(v_issues) > 8:
                    print(f"  ⚠️ ... and {len(v_issues) - 8} more")
            else:
                print("  ✓ YOLO-seg labels OK")

            # Annotated images
            render_combined_image(
                image_frame, result, prompts,
                dirs["annotated"] / f"combined_top{top_n}_sam3.png")
            render_per_prompt_images(
                image_frame, result, prompts, dirs["annotated"])

        log["t_export_s"] = t.elapsed
        log["status"]     = "done"

    except Exception as e:
        import traceback
        log["status"] = "error"
        log["error"]  = traceback.format_exc()
        print(f"  ❌ ERROR: {e}")

    log["t_total_s"]   = time.perf_counter() - wall_start
    log["finished_at"] = datetime.datetime.now().isoformat()

    # Save per-image log
    if log["output_dir"]:
        (Path(log["output_dir"]) / "processing_log.json").write_text(
            json.dumps(log, indent=2))

    print(f"  ✅ Done in {log['t_total_s']:.1f}s  "
          f"(qwen={log['t_qwen_s']:.1f}s  "
          f"sam3={log['t_sam3_s']:.1f}s  "
          f"export={log['t_export_s']:.1f}s)")
    return log

print('✅ Pipeline function ready')

✅ Pipeline function ready


## Step 9 — Run Pipeline

In [35]:
# Collect jobs (flat folder vs USS-PF batch)
if USS_PF_DATASET:
    _pairs = collect_uss_pf_front_rear_jobs(
        Path(INPUT_PATH),
        IMAGE_EXTENSIONS,
        OUTPUT_ROOT,
        max_sequence_folders=MAX_USS_PF_SEQUENCE_FOLDERS,
    )
    if MAX_IMAGE_JOBS is not None:
        _pairs = _pairs[: int(MAX_IMAGE_JOBS)]
    image_paths = [a for a, _ in _pairs]
    job_output_roots = {pipeline_job_id(a): b for a, b in _pairs}
    print(
        f"USS-PF: {len(image_paths)} Front/Rear images under {INPUT_PATH}"
        f" (seq_folder_cap={MAX_USS_PF_SEQUENCE_FOLDERS}, max_jobs={MAX_IMAGE_JOBS})"
    )
else:
    image_paths = collect_images(INPUT_PATH, IMAGE_EXTENSIONS)
    if MAX_IMAGE_JOBS is not None:
        image_paths = image_paths[: int(MAX_IMAGE_JOBS)]
    job_output_roots = None

print(f"Found {len(image_paths)} image(s) in scan (after caps).")

# Checkpoint: merge with on-disk state so we can resume
state = load_merge_pipeline_state(
    PIPELINE_STATE_PATH,
    image_paths,
    OUTPUT_ROOT,
    reset=RESET_PIPELINE_STATE,
    retry_failed=RETRY_FAILED_JOBS,
    persist=not PIPELINE_DRY_RUN,
    job_output_roots=job_output_roots,
)
pending_ids = pending_pipeline_job_ids(state)
n_done = sum(1 for jid in state["order"] if state["jobs"][jid]["status"] == "done")
n_err = sum(1 for jid in state["order"] if state["jobs"][jid]["status"] == "error")
print(
    f"State: {PIPELINE_STATE_PATH}  |  done={n_done}  error={n_err}  "
    f"to_run={len(pending_ids)}  (schema v{state.get('schema_version', 1)})"
)

if PIPELINE_DRY_RUN:
    print("\n=== PIPELINE_DRY_RUN: no inference; no disk writes to pipeline_state.json from merge (persist=False) ===")
    for i, job_id in enumerate(pending_ids[:50], start=1):
        j = state["jobs"][job_id]
        print(f"  [{i}] {Path(j['image_path']).name}")
        print(f"      -> {Path(j['output_root']) / Path(j['image_path']).stem}")
    if len(pending_ids) > 50:
        print(f"  … {len(pending_ids) - 50} more pending")
    all_logs = []
    total_wall = 0.0
    print(f"\nDry run: would process {len(pending_ids)} pending job(s). Set PIPELINE_DRY_RUN = False to execute.")
else:
    all_logs = []
    pipeline_start = time.perf_counter()

    for idx, job_id in enumerate(pending_ids, start=1):
        job = state["jobs"][job_id]
        ip = Path(job["image_path"])
        out_root = Path(job["output_root"])
        job["status"] = "running"
        job["started_at"] = datetime.datetime.now().isoformat()
        job["attempts"] = int(job.get("attempts") or 0) + 1
        save_pipeline_state(PIPELINE_STATE_PATH, state)

        print(f"\n[{idx}/{len(pending_ids)}] Processing {ip.name}")
        log = process_image(
            image_path=ip,
            output_root=out_root,
            top_n=TOP_N,
            bbox_fmt=BBOX_LABEL_FORMAT,
        )
        all_logs.append(log)
        update_job_after_run(state, job_id, log, PIPELINE_STATE_PATH)

    state = json.loads(PIPELINE_STATE_PATH.read_text())
    n_done2 = sum(1 for jid in state["order"] if state["jobs"][jid]["status"] == "done")
    n_err2 = sum(1 for jid in state["order"] if state["jobs"][jid]["status"] == "error")
    total_wall = time.perf_counter() - pipeline_start
    print(f"\n{'='*60}")
    print(f"This run processed {len(all_logs)} image(s) in {total_wall:.1f}s")
    print(f"Checkpoint: done={n_done2}  error={n_err2}  total_tracked={len(state['order'])}")


USS-PF: 10 Front/Rear images under /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/inputs/uss_pf
Found 10 image(s) in scan.
State: /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/outputs/uss_pf/pipeline_state.json  |  done=0  error=0  to_run=10  (schema v1)

[1/10] Processing 20190311_LB_XO2617_012_165618_Front.avi.jpg

IMAGE : 20190311_LB_XO2617_012_165618_Front.avi.jpg
  ⏱  Read image: 0.01s
  Size: 960×640
  ⏱  Qwen inference: 210.03s
  Prompts: ['road', 'hedge', 'car', 'building']
  ⏱  SAM3 inference: 18.70s
  ✓ YOLO-seg labels OK
  ⏱  Export: 0.33s
  ✅ Done in 229.3s  (qwen=210.0s  sam3=18.7s  export=0.3s)

[2/10] Processing 20190311_LB_XO2617_012_165618_Rear.avi.jpg

IMAGE : 20190311_LB_XO2617_012_165618_Rear.avi.jpg
  ⏱  Read image: 0.00s
  Size: 960×640
  ⏱  Qwen inference: 206.72s
  Prompts: ['brick pavement', 'white van', 'red roof', 'tree']
  ⏱  SAM3 inference: 18.86s
  ✓ YOLO-seg labels OK
  ⏱  Export: 0.38s
  ✅ Done in 226.2s  (qwen=206.7s  sam3=18.9s

## Step 10 — Validate YOLO-seg artifacts

Checks every run folder under `OUTPUT_ROOT`: segment `*.txt` lines must be `class x1 y1 x2 y2 …` with normalized coordinates in **[0, 1]** and `class` in **[0, nc − 1]**.

Run this **after** the pipeline (Step 9). Class-id checks use each run’s **`labels/segments/classes.txt`** (open-vocab). Summary `nc` / names in the report use **`open_vocab_state.json`** when present.


In [17]:
# Self-contained: needs Step 3 (OUTPUT_ROOT). No Step 7 required.
from pathlib import Path
import json


def validate_yolo_seg_label_line(line: str, nc: int) -> tuple[bool, str]:
    parts = line.split()
    if len(parts) < 7:
        return False, f"need class + ≥3 (x,y) pairs (got {len(parts)} tokens)"
    try:
        cid = int(float(parts[0]))
    except ValueError:
        return False, "class id must be numeric"
    if cid < 0 or cid >= nc:
        return False, f"class {cid} not in [0, {nc - 1}]"
    try:
        coords = [float(x) for x in parts[1:]]
    except ValueError:
        return False, "non-numeric coordinates"
    if len(coords) % 2 != 0:
        return False, "coordinate count must be even"
    if len(coords) < 6:
        return False, "need ≥ 3 polygon vertices"
    for i, v in enumerate(coords):
        if v < -1e-6 or v > 1.0 + 1e-6:
            return False, f"coord[{i}]={v:g} outside [0,1] (YOLO norm space)"
    return True, "ok"


def _iter_yolo_seg_run_dirs(output_root: Path):
    """Flat OUTPUT_ROOT/<stem>/ or nested OUTPUT_ROOT/<seq>/<stem>/ with labels/segments."""
    for sub in sorted(output_root.iterdir()):
        if not sub.is_dir():
            continue
        if (sub / "labels" / "segments").is_dir():
            yield sub.name, sub
            continue
        for nest in sorted(sub.iterdir()):
            if not nest.is_dir():
                continue
            if (nest / "labels" / "segments").is_dir():
                yield f"{sub.name}/{nest.name}", nest


def _read_seg_nc_from_classes(seg_dir: Path) -> int:
    cls_p = seg_dir / "classes.txt"
    if not cls_p.is_file():
        return 0
    return len([ln for ln in cls_p.read_text().splitlines() if ln.strip()])


def validate_yolo_seg_output_root(output_root: Path, nc: int | None = None) -> dict:
    """Scan run folders; nc None = infer per folder from labels/segments/classes.txt."""
    summary = {"folders": 0, "ok": 0, "with_issues": 0, "details": []}
    for label, run_dir in _iter_yolo_seg_run_dirs(output_root):
        seg_dir = run_dir / "labels" / "segments"
        summary["folders"] += 1
        nc_run = _read_seg_nc_from_classes(seg_dir) if nc is None else nc
        if nc_run <= 0:
            summary["with_issues"] += 1
            summary["details"].append({"folder": label, "issues": ["no classes.txt or empty"]}) 
            continue
        label_txts = [p for p in seg_dir.glob("*.txt") if p.name != "classes.txt"]
        folder_issues: list[str] = []
        if not label_txts:
            folder_issues.append("no *.txt label file (only classes.txt?)")
        for p in label_txts:
            raw = p.read_text().strip()
            if not raw:
                continue
            for li, line in enumerate(raw.splitlines(), 1):
                line = line.strip()
                if not line:
                    continue
                ok, msg = validate_yolo_seg_label_line(line, nc_run)
                if not ok:
                    folder_issues.append(f"{p.name} L{li}: {msg}")
        if folder_issues:
            summary["with_issues"] += 1
            summary["details"].append({"folder": label, "issues": folder_issues})
        else:
            summary["ok"] += 1
    return summary


_vocab_p = OUTPUT_ROOT / "open_vocab_state.json"
_ov_names = []
if _vocab_p.is_file():
    try:
        _ov_names = json.loads(_vocab_p.read_text()).get("id_to_prompt", [])
    except Exception:
        _ov_names = []
report = validate_yolo_seg_output_root(OUTPUT_ROOT, None)
report["nc"] = len(_ov_names)
report["class_names"] = list(_ov_names)
report["note"] = "Per-folder class id checks use each run's classes.txt; nc/names here from global open_vocab_state.json if present."
_vr = OUTPUT_ROOT / "yolo_seg_validation_report.json"
_vr.write_text(json.dumps(report, indent=2))
print(f"Wrote {_vr}")
print(
    f"YOLO-seg scan: {report['folders']} folders with labels/segments, "
    f"{report['ok']} clean, {report['with_issues']} with issues"
)
if report["details"]:
    for d in report["details"][:20]:
        print(f"\n📁 {d['folder']}")
        for iss in d["issues"][:12]:
            print(f"   - {iss}")
    if len(report["details"]) > 20:
        print(f"\n… {len(report['details']) - 20} more folders with issues")
else:
    print("No issues found (or no segment folders yet).")


Wrote /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/outputs/uss_pf/yolo_seg_validation_report.json
YOLO-seg scan: 73 folders with labels/segments, 72 clean, 1 with issues

📁 106346/20190411_LB_XO2617_165134_Front.avi
   - no *.txt label file (only classes.txt?)


## Step 10b — Segmentation label summary

Counts **polygon lines** in `labels/segments/*.txt`. Names for the table come from **`open_vocab_state.json`** on `OUTPUT_ROOT` when available. Writes `yolo_seg_label_summary.json`.


In [28]:
# Self-contained: needs Step 3 (OUTPUT_ROOT).
from pathlib import Path
import json
from collections import Counter


def _iter_yolo_seg_run_dirs(output_root: Path):
    for sub in sorted(output_root.iterdir()):
        if not sub.is_dir():
            continue
        if (sub / "labels" / "segments").is_dir():
            yield sub.name, sub
            continue
        for nest in sorted(sub.iterdir()):
            if not nest.is_dir():
                continue
            if (nest / "labels" / "segments").is_dir():
                yield f"{sub.name}/{nest.name}", nest


def _open_vocab_names(output_root: Path) -> list[str]:
    p = output_root / "open_vocab_state.json"
    if not p.is_file():
        return []
    try:
        return list(json.loads(p.read_text()).get("id_to_prompt", []))
    except Exception:
        return []


def summarize_yolo_seg_labels(output_root: Path, class_names: list[str] | None) -> dict:
    names = list(class_names) if class_names else _open_vocab_names(output_root)
    nc = len(names)
    per_class = Counter()
    total_polygons = 0
    label_files = 0
    unknown_class_ids = Counter()
    run_folders = 0
    per_folder: list[dict] = []

    for label, sub in _iter_yolo_seg_run_dirs(output_root):
        seg_dir = sub / "labels" / "segments"
        run_folders += 1
        folder_ctr = Counter()
        for p in sorted(seg_dir.glob("*.txt")):
            if p.name == "classes.txt":
                continue
            label_files += 1
            raw = p.read_text().strip()
            if not raw:
                continue
            for line in raw.splitlines():
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if not parts:
                    continue
                try:
                    cid = int(float(parts[0]))
                except ValueError:
                    continue
                total_polygons += 1
                if nc > 0 and 0 <= cid < nc:
                    per_class[cid] += 1
                    folder_ctr[cid] += 1
                elif nc <= 0:
                    per_class[cid] += 1
                    folder_ctr[cid] += 1
                else:
                    unknown_class_ids[cid] += 1
        per_folder.append(
            {
                "folder": label,
                "polygons": sum(folder_ctr.values()),
                "by_class_id": {str(k): folder_ctr[k] for k in sorted(folder_ctr)},
            }
        )

    max_cid = max(per_class.keys(), default=-1)
    table_nc = max(nc, max_cid + 1) if max_cid >= 0 else nc
    per_class_named = {}
    for cid in range(table_nc):
        nm = names[cid] if cid < len(names) else f"(id {cid})"
        per_class_named[str(cid)] = {"name": nm, "count": per_class[cid]}
    unknown_named = {
        f"out_of_range_{cid}": {"name": f"id {cid} (not in 0..{nc - 1})", "count": cnt}
        for cid, cnt in sorted(unknown_class_ids.items())
    }

    return {
        "run_folders_with_segments": run_folders,
        "segment_label_files": label_files,
        "total_polygons": total_polygons,
        "per_class": {**per_class_named, **unknown_named},
        "per_folder": per_folder,
    }


seg_summary = summarize_yolo_seg_labels(OUTPUT_ROOT, None)
_seg_out = OUTPUT_ROOT / "yolo_seg_label_summary.json"
_seg_out.write_text(json.dumps(seg_summary, indent=2))
print(f"Wrote {_seg_out}")
print(
    f"Runs with segments: {seg_summary['run_folders_with_segments']}  |  "
    f"label files: {seg_summary['segment_label_files']}  |  "
    f"total polygons: {seg_summary['total_polygons']}"
)
print()
print(f"{'Class id':<8} {'Name':<36} {'Count':>8}")
print("-" * 56)
_row_keys = sorted(
    (int(k) for k in seg_summary["per_class"] if k.isdigit()),
    key=int,
)
for cid in _row_keys:
    row = seg_summary["per_class"][str(cid)]
    print(f"{cid:<8} {row['name'][:36]:<36} {row['count']:>8}")
for k, v in seg_summary["per_class"].items():
    if k.startswith("out_of_range_"):
        print(f"{'—':<8} {v['name'][:36]:<36} {v['count']:>8}")
print()
print("Per run folder (first 25):")
for row in seg_summary["per_folder"][:25]:
    bc = row.get("by_class_id") or {}
    detail = ", ".join(f"c{k}:{bc[k]}" for k in sorted(bc, key=int))
    print(f"  {row['folder'][:48]:<48}  polys={row['polygons']:>4}  [{detail}]")
if len(seg_summary["per_folder"]) > 25:
    print(f"  … {len(seg_summary['per_folder']) - 25} more folders")


Wrote /Users/ral3ply/Documents/GitHub/visual-search/auto-labeler/outputs/uss_pf/yolo_seg_label_summary.json
Runs with segments: 0  |  label files: 0  |  total polygons: 0

Class id Name                                    Count
--------------------------------------------------------
0        road                                        0
1        pavement                                    0
2        building                                    0
3        wall                                        0
4        car                                         0
5        tree                                        0
6        sky                                         0
7        person                                      0
8        fence                                       0
9        pole                                        0
10       sign                                        0
11       barrier                                     0
12       railing                                     0
1

## Step 11 — Summary Report

In [19]:
done = [l for l in all_logs if l["status"] == "done"]
errs = [l for l in all_logs if l["status"] == "error"]

summary = {
    "generated_at":         datetime.datetime.now().isoformat(),
    "input_path":           str(INPUT_PATH),
    "total_images":         len(all_logs),
    "succeeded":            len(done),
    "failed":               len(errs),
    "total_wall_time_s":    round(total_wall, 2),
    "avg_time_per_image_s": round(total_wall / max(len(all_logs), 1), 2),
    "avg_qwen_time_s":      round(sum(l["t_qwen_s"]   for l in done) / max(len(done),1), 2),
    "avg_sam3_time_s":      round(sum(l["t_sam3_s"]   for l in done) / max(len(done),1), 2),
    "avg_export_time_s":    round(sum(l["t_export_s"] for l in done) / max(len(done),1), 2),
    "per_image":            all_logs,
}

summary_path = OUTPUT_ROOT / "summary_report.json"
summary_path.write_text(json.dumps(summary, indent=2))

# Print table
print(f"{'─'*70}")
print(f"{'Image':<30} {'Qwen(s)':>8} {'SAM3(s)':>9} {'Total(s)':>9} {'Status':>8}")
print(f"{'─'*70}")
for l in all_logs:
    name = Path(l['image_path']).name[:28]
    print(f"{name:<30} "
          f"{l['t_qwen_s']:>8.1f} {l['t_sam3_s']:>9.1f} "
          f"{l['t_total_s']:>9.1f} {l['status']:>8}")
print(f"{'─'*70}")
print(f"{'TOTAL':<30} {'':>8} {'':>9} {summary['total_wall_time_s']:>9.1f}")
print(f"\nSummary saved → {summary_path}")

if errs:
    print(f"\n⚠️  {len(errs)} image(s) failed:")
    for l in errs:
        print(f"  {l['image_path']}\n  {l['error'][:200]}")

──────────────────────────────────────────────────────────────────────
Image                           Qwen(s)   SAM3(s)  Total(s)   Status
──────────────────────────────────────────────────────────────────────
20190311_LB_XO2617_012_16561      558.7      20.6     581.1     done
20190311_LB_XO2617_012_16561      625.1      19.0     645.0     done
20190411_LB_XO2617_171125_Fr      641.1      19.3     661.2     done
20190411_LB_XO2617_171125_Re      651.4      19.6     672.0     done
20190307_LB_XO2617_003_12570      636.4      18.5     656.1     done
20190307_LB_XO2617_003_12570      549.2      19.9     570.0     done
20190329_LB_XO2617_092212_Fr      613.0      19.4     633.1     done
20190329_LB_XO2617_092212_Re      550.4      21.3     572.7     done
20190411_LB_XO2617_110929_Fr      489.5      18.8     509.4     done
20190411_LB_XO2617_110929_Re      512.9      19.0     532.7     done
20190411_LB_XO2617_165352_Fr      495.8      19.0     515.5     done
20190411_LB_XO2617_165352_Re  